# 🏏 IPL Win Probability Predictor: End-to-End Machine Learning Pipeline
### **Non-Teaching Credit Course (NTCC) Major Project & Portfolio**

## 📌 Project Overview
The **Indian Premier League (IPL)** is one of the most dynamic T20 cricket leagues globally. In limited-overs cricket, especially during the second innings run chase, the momentum shifts with every single delivery. 

This notebook demonstrates a complete end-to-end Machine Learning solution to calculate the real-time **Win Probability** of the chasing team versus the defending team based on in-game state parameters:
- **Target Score & Runs Left**
- **Balls Remaining**
- **Wickets in Hand**
- **Current Run Rate (CRR)**
- **Required Run Rate (RRR)**
- **Batting Team, Bowling Team & Match Venue (City)**

## 1. Imports and Environment Setup

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, log_loss

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries successfully imported!")

## 2. Dataset Loading
We load two datasets:
1. `matches.csv`: Contains match metadata (season, teams, toss, venue, winner).
2. `deliveries.csv`: Contains ball-by-ball delivery information (inning, over, ball, batsman, bowler, total_runs, dismissals).

In [ ]:
data_dir = '../data'
matches_path = os.path.join(data_dir, 'matches.csv')
deliveries_path = os.path.join(data_dir, 'deliveries.csv')

# Check if Kaggle datasets exist; if not, use sample dataset
if os.path.exists(matches_path) and os.path.exists(deliveries_path):
    matches = pd.read_csv(matches_path)
    deliveries = pd.read_csv(deliveries_path)
    print(f"Loaded matches: {matches.shape}, deliveries: {deliveries.shape}")
else:
    print("Kaggle files not found. Please place matches.csv & deliveries.csv in data/ or run train_model.py to generate sample data.")

## 3. Data Cleaning & Normalization
- Standardize renamed franchises (`Delhi Daredevils` -> `Delhi Capitals`, `Kings XI Punjab` -> `Punjab Kings`, `Deccan Chargers` -> `Sunrisers Hyderabad`).
- Filter for currently active franchises.
- Eliminate rain-affected (DLS) matches to avoid skewed run-rate targets.

In [ ]:
ACTIVE_TEAMS = [
    'Chennai Super Kings',
    'Delhi Capitals',
    'Gujarat Titans',
    'Kolkata Knight Riders',
    'Lucknow Super Giants',
    'Mumbai Indians',
    'Punjab Kings',
    'Rajasthan Royals',
    'Royal Challengers Bangalore',
    'Sunrisers Hyderabad'
]

TEAM_MAPPINGS = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Royal Challengers Bengaluru': 'Royal Challengers Bangalore'
}

## 4. Feature Engineering for 2nd Innings Run Chases
For each match's 2nd innings, we compute:
1. **Target Score (`total_runs_x`)**: Runs scored in 1st innings + 1
2. **Current Score**: Cumulative sum of runs scored up to that delivery
3. **Runs Left**: `Target Score - Current Score`
4. **Balls Left**: `120 - Balls Bowled`
5. **Wickets Left**: `10 - Cumulative Dismissals`
6. **Current Run Rate (CRR)**: `(Current Score * 6) / Balls Bowled`
7. **Required Run Rate (RRR)**: `(Runs Left * 6) / Balls Left`
8. **Result**: Binary target (1 if Batting Team won the match, 0 if Bowling Team won)

## 5. Machine Learning Pipeline Architecture
We use a Scikit-Learn **Pipeline**:
1. **ColumnTransformer + OneHotEncoder**: Encodes `batting_team`, `bowling_team`, and `city`.
2. **LogisticRegression(solver='liblinear')**: Why Logistic Regression? 
   - Logistic Regression models the log-odds of winning linearly with respect to features and applies the sigmoid function:
     $$P(Y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \sum \beta_i X_i)}}$$
   - Tree-based models (e.g. standard Decision Trees or uncalibrated Random Forests) output step-wise, jagged probabilities. Logistic Regression gives smooth, monotonically calibrated probabilities that reflect realistic live sports win probabilities.

In [ ]:
# Preprocessing pipeline
trf = ColumnTransformer([
    ('encoder', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), ['batting_team', 'bowling_team', 'city'])
], remainder='passthrough')

# Model pipeline
pipe = Pipeline(steps=[
    ('step1', trf),
    ('step2', LogisticRegression(solver='liblinear', random_state=42, max_iter=1000))
])

print(pipe)

## 6. Model Evaluation and Calibration
We evaluate the model on an unseen 20% stratified test set using:
- **Accuracy Score**
- **Log Loss (Cross-Entropy Loss)**: Measures probability calibration
- **ROC-AUC Score**: Measures discrimination between wins and losses

## 7. Model Export for Streamlit
Save the fitted pipeline to `../models/pipe.pkl` using `pickle` so that the Streamlit application `app.py` can load it instantly.

In [ ]:
# Example export code:
# with open('../models/pipe.pkl', 'wb') as f:
#     pickle.dump(pipe, f)
# print("Pipeline serialized successfully to models/pipe.pkl!")